In [3]:
import pandas as pd
import os


github_raw_link = "https://raw.githubusercontent.com/ahmedtarek-5/ml-1/refs/heads/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(github_raw_link)
print("Data loaded successfully from GitHub!")
print(f"Total rows: {len(df)}")

os.makedirs("work/outputs", exist_ok=True)
print("'work/outputs' directory is ready.")

Data loaded successfully from GitHub!
Total rows: 30000
'work/outputs' directory is ready.


# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedtarek-5/ml-1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes


**The Baseline Rule (Plain Words):**
We prioritize pages for review based on three observable, historical signals:
1. **Staleness:** The content is old (age > 180 days). (+2 points)
2. **Decline:** The page is showing a downward trend in performance. (+3 points)
3. **Baseline Visibility:** The page has historically achieved at least the median impressions, meaning it has proven potential worth saving. (+1 point)

**Action Labels based on Score:**
- Score >= 4: **"Refresh"** (High priority for immediate human review)
- Score 2 to 3: **"Review"** (Medium priority, check for specific issues)
- Score 0 to 1: **"Monitor"** (Low priority, no immediate action needed)

**Reason Codes Generated:**
- `"Stale + Declining + Visible"` (Score: 6)
- `"Stale + Declining"` (Score: 5)
- `"Declining + Visible"` (Score: 4)
- `"Stale + Visible"` (Score: 3)
- `"Declining only"` (Score: 3)
- `"Stale only"` (Score: 2)
- `"Visible only"` (Score: 1)
- `"No signal"` (Score: 0)

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)
This cell applies the baseline rule to the entire dataset, calculates the score and reason code for each page, sorts them in descending order, and writes the final ranked queue to `work/outputs/baseline_action_score.csv`.

In [4]:
import pandas as pd
import numpy as np

# Calculate median impressions for the visibility threshold
median_impressions = df['impressions_90d'].median() if 'impressions_90d' in df.columns else 0

def calculate_baseline_score(row):
    score = 0
    reasons = []

    # 1. Check Staleness
    if 'content_age_days' in row and pd.notna(row['content_age_days']) and row['content_age_days'] > 180:
        score += 2
        reasons.append("Stale")

    # 2. Check Decline
    if 'trend_direction' in row and str(row['trend_direction']).strip().lower() == 'down':
        score += 3
        reasons.append("Declining")

    # 3. Check Baseline Visibility
    if 'impressions_90d' in row and pd.notna(row['impressions_90d']) and row['impressions_90d'] > median_impressions:
        score += 1
        reasons.append("Visible")

    # Generate Reason Code
    reason_code = " + ".join(reasons) if reasons else "No signal"

    # Determine Action Label
    if score >= 4:
        action = "Refresh"
    elif score >= 2:
        action = "Review"
    else:
        action = "Monitor"

    return pd.Series([score, reason_code, action])

# Apply the rule to the dataframe
df[['baseline_score', 'reason_code', 'action_label']] = df.apply(calculate_baseline_score, axis=1)

# Sort by score descending to create the ranked queue
df_ranked = df.sort_values(by='baseline_score', ascending=False).reset_index(drop=True)

# Define the columns to save in the CSV
cols_to_save = ['content_id', 'baseline_score', 'reason_code', 'action_label', 'content_age_days', 'trend_direction', 'impressions_90d']
available_cols = [col for col in cols_to_save if col in df_ranked.columns]

# Write to CSV
output_path = "work/outputs/baseline_action_score.csv"
df_ranked[available_cols].to_csv(output_path, index=False)

print(f"Ranked queue built successfully!")
print(f"Saved to: {output_path}")
print("\nScore Distribution:")
print(df_ranked['action_label'].value_counts())

Ranked queue built successfully!
Saved to: work/outputs/baseline_action_score.csv

Score Distribution:
action_label
Review     12770
Refresh    12656
Monitor     4574
Name: count, dtype: int64


## 3. Top-20 review

Below is an automated review of the top 20 pages recommended for action. For each, we state the recommended action, the reason code, a confidence note based on data completeness, and what would make this recommendation wrong (the failure mode).

In [5]:
# Get the top 20 rows
top_20 = df_ranked.head(20).copy()

# Add programmatic review columns
def add_review_notes(row):
    # Confidence based on data completeness
    has_impressions = pd.notna(row.get('impressions_90d'))
    has_age = pd.notna(row.get('content_age_days'))
    confidence = "High" if (has_impressions and has_age) else "Low (Missing data)"

    # What would make it wrong? (Failure mode)
    if row['action_label'] == 'Refresh' and row.get('impressions_90d', 0) < 50:
        failure_mode = "Page might be naturally dead/unindexable, not just stale."
    elif row['action_label'] == 'Review' and row.get('content_age_days', 0) < 90:
        failure_mode = "Page is too new; decline might be normal settling, not a content issue."
    else:
        failure_mode = "External factors (e.g., algorithm update, seasonality) caused the drop, not content quality."

    return pd.Series([confidence, failure_mode])

top_20[['Confidence', 'What_would_make_it_wrong']] = top_20.apply(add_review_notes, axis=1)

# Display the review table
review_cols = ['content_id', 'action_label', 'reason_code', 'Confidence', 'What_would_make_it_wrong']
available_review_cols = [col for col in review_cols if col in top_20.columns]

print("🔍 Top-20 Baseline Recommendations Review:")
display(top_20[available_review_cols])

🔍 Top-20 Baseline Recommendations Review:


,content_id,action_label,reason_code,Confidence,What_would_make_it_wrong
0,content_304f48230142,Refresh,Stale + Declining + Visible,High,"External factors (e.g., algorithm update, seas..."
1,content_d99b7a2d90ca,Refresh,Stale + Declining + Visible,High,"External factors (e.g., algorithm update, seas..."
2,content_24398d5d8731,Refresh,Stale + Declining + Visible,High,"External factors (e.g., algorithm update, seas..."
3,content_ac9af6d6dad9,Refresh,Stale + Declining + Visible,High,"External factors (e.g., algorithm update, seas..."
4,content_0e23e310d404,Refresh,Stale + Declining + Visible,High,"External factors (e.g., algorithm update, seas..."
5,content_7ea135180dd9,Refresh,Stale + Declining + Visible,High,"External factors (e.g., algorithm update, seas..."
6,content_1db0d204d42f,Refresh,Stale + Declining + Visible,High,"External factors (e.g., algorithm update, seas..."
7,content_bdee2164f576,Refresh,Stale + Declining + Visible,High,"External factors (e.g., algorithm update, seas..."
8,content_3b7c0934332e,Refresh,Stale + Declining + Visible,High,"External factors (e.g., algorithm update, seas..."
9,content_e540f63bc134,Refresh,Stale + Declining + Visible,High,"External factors (e.g., algorithm update, seas..."


## 4. Weak picks + leakage check

**Weak Picks Analysis:**
Some pages in the "Refresh" category might have high age and a "down" trend, but extremely low historical impressions (e.g., < 50). These are weak picks because the content might have never been valuable or indexed properly in the first place. A human reviewer should verify indexation status before spending time on them.

**Leakage Check Confirmation:**
- No product flags (e.g., `is_refreshed`, `manual_override`) were used as features.
- No future-window metrics (e.g., `traffic_next_month`, `post_update_ctr`) are present in the scoring logic.
- All features (`content_age_days`, `trend_direction`, `impressions_90d`) are strictly historica

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.